### Loading Data and Libraries

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("../data/processed/pricing_analyst_cleaned.csv")
FIGURES_DIR = Path("../outputs/figures")
TABLES_DIR = Path("../outputs/tables")

df_analysis = pd.read_csv(DATA_PATH)

date_columns = ["offerdate", "purchase_date"]
for col in date_columns:
    df_analysis[col] = pd.to_datetime(df_analysis[col], errors="coerce")

print(df_analysis.shape)
df_analysis.info()
display(df_analysis.head())

(8865, 24)
<class 'pandas.DataFrame'>
RangeIndex: 8865 entries, 0 to 8864
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   offerdate                      8865 non-null   datetime64[us]
 1   sold_premium                   1991 non-null   float64       
 2   offered_premium                8865 non-null   float64       
 3   purchase_price                 8864 non-null   float64       
 4   purchase_date                  8865 non-null   datetime64[us]
 5   item_age                       8862 non-null   float64       
 6   pricing_point                  8865 non-null   str           
 7   predictedconversionrate        8865 non-null   float64       
 8   plan_flag                      8865 non-null   int64         
 9   plan_count                     8824 non-null   float64       
 10  plansactive_lastyear_count     8824 non-null   float64       
 11  planscancelled_la

,offerdate,sold_premium,offered_premium,purchase_price,purchase_date,item_age,pricing_point,predictedconversionrate,plan_flag,plan_count,...,price_diff,ismodel,sale_flag,base_rate,manufacturerbrandname_enc,itemcategoryname_enc,itemsupercategorycode_enc,invalid_price_flag,price_diff_check,price_diff_matches
0,2023-03-17,NaN,32.64,89.99,2023-03-16,1.0,@22%,0.15,0,0.0,...,-0.058824,yes,0,34.68,56,35,14,False,-1.981176,False
1,2023-03-01,69.72,69.72,329.00,2022-12-24,67.0,@22%,0.81,1,5.0,...,-0.023529,yes,1,71.40,123,34,3,False,-1.656471,False
2,2023-04-12,NaN,48.24,249.00,2023-04-05,7.0,@23%,0.08,0,0.0,...,0.210843,yes,0,39.84,7,16,12,False,8.189157,False
3,2023-03-09,NaN,91.92,746.42,2021-03-09,730.0,@23%,0.32,0,0.0,...,0.298305,yes,0,70.80,107,36,4,False,20.821695,False
4,2023-03-18,NaN,89.64,493.98,2023-03-18,0.0,@22%,0.25,1,1.0,...,0.299130,yes,0,69.00,57,36,4,False,20.340870,False


## Pricing Performance Summary Table

In [18]:
strategy_order = ["ASIS FEE", "@22%", "@23%"]

summary_table = (
    df_analysis
    .groupby("pricing_point")
    .agg(
        no_of_offers=("sale_flag", "size"),
        avg_base_rate=("base_rate", "mean"),
        avg_offered_premium=("offered_premium", "mean"),
        avg_sold_premium=("sold_premium", "mean"),
        avg_price_increase=("price_diff", "mean"),
        conversion_rate=("sale_flag", "mean")
    )
    .reindex(strategy_order)
)

summary_table["conversion_rate_pct"] = (summary_table["conversion_rate"] * 100).round(2)
display(summary_table)
# summary_table.to_csv("../outputs/tables/summary_table.csv", index=True)
summary_table.reset_index().to_csv("../outputs/tables/summary_table.csv", index=False)

,no_of_offers,avg_base_rate,avg_offered_premium,avg_sold_premium,avg_price_increase,conversion_rate,conversion_rate_pct
pricing_point,,,,,,,
ASIS FEE,1115,48.973668,48.973668,49.572923,0.000000,0.233184,23.32
@22%,3923,49.200958,55.212603,55.750694,0.124322,0.220240,22.02
@23%,3827,49.014236,51.976650,51.482768,0.059686,0.226548,22.65


### Statistical Test : do conversion rates differ?

Conversion is binary therefore, we do a chi-squared test across the three groups

In [11]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df_analysis["pricing_point"], df_analysis["sale_flag"])
display(contingency)

chi2, p_value1, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value1:.6f}")

sale_flag,0,1
pricing_point,,
@22%,3059,864
@23%,2960,867
ASIS FEE,855,260


Chi-square statistic: 0.98
Degrees of freedom: 2
p-value: 0.611569


### Statistical Test : do the premiums differ?

Using t-tests on each pair

In [7]:
from scipy.stats import ttest_ind

asis_premium = df_analysis.loc[df_analysis["pricing_point"] == "ASIS FEE", "offered_premium"]
p22_premium = df_analysis.loc[df_analysis["pricing_point"] == "@22%", "offered_premium"]
p23_premium = df_analysis.loc[df_analysis["pricing_point"] == "@23%", "offered_premium"]

t_stat_asis_22, p_asis_22 = ttest_ind(asis_premium, p22_premium, equal_var=False)
t_stat_asis_23, p_asis_23 = ttest_ind(asis_premium, p23_premium, equal_var=False)
t_stat_22_23, p_22_23 = ttest_ind(p22_premium, p23_premium, equal_var=False)

print(f"ASIS vs @22%: t = {t_stat_asis_22:.2f}, p = {p_asis_22:.6f}")
print(f"ASIS vs @23%: t = {t_stat_asis_23:.2f}, p = {p_asis_23:.6f}")
print(f"@22% vs @23%: t = {t_stat_22_23:.2f}, p = {p_22_23:.6f}")

ASIS vs @22%: t = -12.80, p = 0.000000
ASIS vs @23%: t = -6.17, p = 0.000000
@22% vs @23%: t = 8.66, p = 0.000000


### Staistical Test : does the price_diff differ?

Also using t-tests

In [8]:
asis_pricediff = df_analysis.loc[df_analysis["pricing_point"] == "ASIS FEE", "price_diff"]
p22_pricediff = df_analysis.loc[df_analysis["pricing_point"] == "@22%", "price_diff"]
p23_pricediff = df_analysis.loc[df_analysis["pricing_point"] == "@23%", "price_diff"]

t_stat, p_value = ttest_ind(p22_pricediff, p23_pricediff, equal_var=False)
print(f"@22% vs @23% price_diff: t = {t_stat:.2f}, p = {p_value:.6f}")

@22% vs @23% price_diff: t = 21.72, p = 0.000000


In [12]:
stat_test_results = pd.DataFrame({
    "comparison": [
        "ASIS vs @22% (offered_premium)",
        "ASIS vs @23% (offered_premium)",
        "@22% vs @23% (offered_premium)",
        "@22% vs @23% (price_diff)",
        "All strategies (conversion, chi-square)"
    ],
    "test": ["Welch t-test", "Welch t-test", "Welch t-test", "Welch t-test", "Chi-square"],
    "statistic": [t_stat_asis_22, t_stat_asis_23, t_stat_22_23, t_stat, chi2],
    "p_value": [p_asis_22, p_asis_23, p_22_23, p_value, p_value1]
})

display(stat_test_results)
stat_test_results.to_csv("../outputs/tables/statistical_tests.csv", index=False)

,comparison,test,statistic,p_value
0,ASIS vs @22% (offered_premium),Welch t-test,-12.797737,3.465742e-36
1,ASIS vs @23% (offered_premium),Welch t-test,-6.169870,8.160815e-10
2,@22% vs @23% (offered_premium),Welch t-test,8.657218,5.815867e-18
3,@22% vs @23% (price_diff),Welch t-test,21.715181,1.515599e-101
4,"All strategies (conversion, chi-square)",Chi-square,0.983456,6.115687e-01
